# DDoS Visual Forensic Analysis: FiveM Server Attack
This notebook performs a deep visual analysis of a pcapng capture from a FiveM server under a DoS/DDoS attack.

### Setup Instructions:
1. Ensure your pcap file is at `/content/drive/MyDrive/pcap/17-08-2022.pcapng`.
2. **Important:** To use Visual Module 4 (Geo-mapping), you must upload the `GeoLite2-City.mmdb` file to your Google Drive and update the path in the setup cell below. You can obtain this from [MaxMind](https://dev.maxmind.com/geoip/geolite2-free-geolocation-data).

In [ ]:
!pip install dpkt pandas numpy plotly folium geoip2 tqdm branca ipwhois kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.0/195.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.3/101.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 13.4 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
import numpy as np
from google.colab import drive

# --- CONFIGURATION ---
PCAP_PATH = '/content/drive/MyDrive/pcap/17-08-2022.pcapng'
BASE_OUTPUT_DIR = '/content/drive/MyDrive/DDoS_Visual_Analysis/'
GEOIP_DB_PATH = '/content/drive/MyDrive/GeoLite2-City.mmdb' # Update this path

# --- DIRECTORY SETUP ---
subfolders = [
    'charts/timeline/',
    'charts/protocol_ports/',
    'charts/packet_behavior/',
    'charts/geo_map/',
    'data/'
]

def setup_directories(base, folders):
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    for folder in folders:
        full_path = os.path.join(base, folder)
        os.makedirs(full_path, exist_ok=True)
        print(f"Verified folder: {full_path}")

setup_directories(BASE_OUTPUT_DIR, subfolders)

# Paths for data persistence
PARSED_CSV_PATH = os.path.join(BASE_OUTPUT_DIR, 'data/parsed_flows.csv')
ENRICHED_CSV_PATH = os.path.join(BASE_OUTPUT_DIR, 'data/ip_geo_enriched.csv')

Verified folder: /content/drive/MyDrive/DDoS_Visual_Analysis/charts/timeline/
Verified folder: /content/drive/MyDrive/DDoS_Visual_Analysis/charts/protocol_ports/
Verified folder: /content/drive/MyDrive/DDoS_Visual_Analysis/charts/packet_behavior/
Verified folder: /content/drive/MyDrive/DDoS_Visual_Analysis/charts/geo_map/
Verified folder: /content/drive/MyDrive/DDoS_Visual_Analysis/data/


In [ ]:
!wget -O /content/GeoLite2-City.mmdb.gz https://cdn.jsdelivr.net/npm/geolite2-city/GeoLite2-City.mmdb.gz
!gunzip -f /content/GeoLite2-City.mmdb.gz
GEOIP_DB_PATH = '/content/GeoLite2-City.mmdb'
print(f"GeoIP Database ready at: {GEOIP_DB_PATH}")

--2026-04-08 03:46:03--  https://cdn.jsdelivr.net/npm/geolite2-city/GeoLite2-City.mmdb.gz
Resolving cdn.jsdelivr.net (cdn.jsdelivr.net)... 151.101.1.229, 151.101.65.229, 151.101.129.229, ...
Connecting to cdn.jsdelivr.net (cdn.jsdelivr.net)|151.101.1.229|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 31500826 (30M) [application/gzip]
Saving to: ‘/content/GeoLite2-City.mmdb.gz’

/content/GeoLite2-C 100%[===================>]  30.04M   179MB/s    in 0.2s    

2026-04-08 03:46:04 (179 MB/s) - ‘/content/GeoLite2-City.mmdb.gz’ saved [31500826/31500826]

GeoIP Database ready at: /content/GeoLite2-City.mmdb


In [ ]:
import dpkt
import socket
import csv
from datetime import datetime
from tqdm import tqdm

def parse_pcap(pcap_path, output_csv):
    print(f"Starting stream-parse of {pcap_path}...")

    with open(pcap_path, 'rb') as f:
        try:
            pcap = dpkt.pcapng.Reader(f)
        except:
            f.seek(0)
            pcap = dpkt.pcap.Reader(f)

        with open(output_csv, 'w', newline='') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow([
                'epoch_time', 'timestamp', 'src_ip', 'dst_ip',
                'src_port', 'dst_port', 'protocol', 'ttl',
                'packet_length', 'payload_size', 'tcp_flags', 'is_fragmented'
            ])

            for ts, buf in tqdm(pcap, desc="Parsing Packets"):
                try:
                    eth = dpkt.ethernet.Ethernet(buf)
                    if not isinstance(eth.data, dpkt.ip.IP):
                        continue

                    ip = eth.data
                    src_ip = socket.inet_ntoa(ip.src)
                    dst_ip = socket.inet_ntoa(ip.dst)

                    protocol = "OTHER"
                    src_port = 0
                    dst_port = 0
                    tcp_flags = ""
                    payload_size = len(ip.data)

                    if isinstance(ip.data, dpkt.tcp.TCP):
                        protocol = "TCP"
                        src_port = ip.data.sport
                        dst_port = ip.data.dport
                        flags = ip.data.flags
                        flag_list = []
                        if flags & dpkt.tcp.TH_SYN: flag_list.append('SYN')
                        if flags & dpkt.tcp.TH_ACK: flag_list.append('ACK')
                        if flags & dpkt.tcp.TH_RST: flag_list.append('RST')
                        if flags & dpkt.tcp.TH_FIN: flag_list.append('FIN')
                        if flags & dpkt.tcp.TH_PUSH: flag_list.append('PSH')
                        if flags & dpkt.tcp.TH_URG: flag_list.append('URG')
                        tcp_flags = "|".join(flag_list)
                    elif isinstance(ip.data, dpkt.udp.UDP):
                        protocol = "UDP"
                        src_port = ip.data.sport
                        dst_port = ip.data.dport
                    elif isinstance(ip.data, dpkt.icmp.ICMP):
                        protocol = "ICMP"

                    is_fragmented = (ip.off & dpkt.ip.IP_MF) != 0 or (ip.off & dpkt.ip.IP_OFFMASK) != 0
                    dt_obj = datetime.fromtimestamp(ts)

                    writer.writerow([
                        ts, dt_obj.strftime('%Y-%m-%d %H:%M:%S.%f'),
                        src_ip, dst_ip, src_port, dst_port,
                        protocol, ip.ttl, len(buf), payload_size,
                        tcp_flags, is_fragmented
                    ])
                except:
                    continue

    print(f"\nParsing complete. Saved to: {output_csv}")

if not os.path.exists(PARSED_CSV_PATH):
    parse_pcap(PCAP_PATH, PARSED_CSV_PATH)
else:
    print("Parsed CSV already exists. Skipping parsing step.")

Starting stream-parse of /content/drive/MyDrive/pcap/17-08-2022.pcapng...


Parsing Packets: 0it [00:00, ?it/s]/usr/local/lib/python3.12/dist-packages/dpkt/ip.py:140: UserWarning: IP.off is deprecated
  deprecation_warning("IP.off is deprecated")
Parsing Packets: 18307235it [13:53, 21969.23it/s]


Parsing complete. Saved to: /content/drive/MyDrive/DDoS_Visual_Analysis/data/parsed_flows.csv


## Visual Module 1: Attack Timeline & Traffic Spikes
This module visualizes the intensity of the attack over time. Forensic analysts look for:
1. **PPS/Mbps Spikes**: To determine the exact start/stop time and the scale of the flood.
2. **Protocol Transitions**: To see if the attacker switched from one method (e.g., UDP flood) to another (e.g., TCP SYN flood).
3. **Attack Patterns**: The heatmap reveals if the attack is continuous or pulsing/bursty.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import os

def run_module_1(csv_path, output_dir):
    df = pd.read_csv(csv_path)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    # Using 's' instead of 'S' to avoid deprecation warning
    df['second'] = df['timestamp'].dt.floor('s')

    # --- CHART 1A: ATTACK PULSE TIMELINE ---
    try:
        timeline = df.groupby('second').agg({'packet_length': ['count', 'sum']})
        timeline.columns = ['pps', 'bps']
        timeline['mbps'] = (timeline['bps'] * 8) / 1000000
        timeline = timeline.reset_index()

        baseline_pps = timeline['pps'].quantile(0.1)
        peak_idx = timeline['pps'].idxmax()
        peak_time = timeline.loc[peak_idx, 'second']
        peak_pps = timeline.loc[peak_idx, 'pps']

        fig1a = make_subplots(specs=[[{"secondary_y": True}]])
        fig1a.add_trace(go.Scatter(x=timeline['second'], y=timeline['pps'], name="PPS (Packets/Sec)", line=dict(color='red')), secondary_y=False)
        fig1a.add_trace(go.Scatter(x=timeline['second'], y=timeline['mbps'], name="Mbps (Megabits/Sec)", line=dict(color='blue')), secondary_y=True)

        fig1a.add_hline(y=baseline_pps, line_dash="dash", line_color="green", annotation_text="Baseline")
        fig1a.add_annotation(x=peak_time, y=peak_pps, text=f"PEAK: {peak_pps} PPS", showarrow=True, arrowhead=1, bgcolor="red")

        fig1a.update_layout(title="DDoS Attack Traffic Timeline — FiveM Server", template="plotly_dark", xaxis_rangeslider_visible=True)
        fig1a.write_html(os.path.join(output_dir, 'charts/timeline/attack_timeline.html'))
        fig1a.show()
    except Exception as e: print(f"Error Chart 1A: {e}")

    # --- CHART 1B: PROTOCOL TRAFFIC OVER TIME ---
    try:
        proto_time = df.groupby(['second', 'protocol']).size().unstack(fill_value=0).reset_index()
        # Dynamically identify columns present (excluding 'second')
        cols = [c for c in proto_time.columns if c != 'second']
        fig1b = px.area(proto_time, x='second', y=cols,
                        title="Protocol Breakdown Over Time", template="plotly_dark",
                        color_discrete_map={'TCP':'blue', 'UDP':'orange', 'ICMP':'red', 'OTHER':'grey'})
        fig1b.write_html(os.path.join(output_dir, 'charts/timeline/protocol_stacked_area.html'))
        fig1b.show()
    except Exception as e: print(f"Error Chart 1B: {e}")

    # --- CHART 1C: ATTACK WAVE HEATMAP ---
    try:
        df['minute'] = df['timestamp'].dt.strftime('%H:%M')
        df['sec_in_min'] = df['timestamp'].dt.second
        heatmap_data = df.groupby(['minute', 'sec_in_min']).size().unstack(fill_value=0)

        fig1c = px.imshow(heatmap_data, labels=dict(x="Second of Minute", y="Minute", color="Packets"),
                          title="Attack Wave Heatmap — Packets Per Second Grid",
                          color_continuous_scale='Reds', template="plotly_dark")
        fig1c.write_html(os.path.join(output_dir, 'charts/timeline/attack_heatmap.html'))
        fig1c.show()
    except Exception as e: print(f"Error Chart 1C: {e}")

run_module_1(PARSED_CSV_PATH, BASE_OUTPUT_DIR)

## Visual Module 2: Protocol & Port Breakdown
This module identifies the primary 'targets' of the attack.
- **Sunburst & Bars**: Reveal if the attack was a standard FiveM attack (Port 30120) or if other services like SSH or DNS were targeted.
- **Scatter Plot**: Identifies randomized source port attacks (common in spoofed UDP floods).
- **Animated Race**: Shows the 'evolution' of the attack as the attacker potentially switches protocols.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os

def run_module_2(csv_path, output_dir):
    df = pd.read_csv(csv_path)

    # --- CHART 2A: PROTOCOL DISTRIBUTION (SUNBURST) ---
    try:
        # Group by protocol and top ports
        sun_data = df.groupby(['protocol', 'dst_port']).size().reset_index(name='counts')
        # Filter for top 10 ports per protocol for readability
        sun_data = sun_data.groupby('protocol').apply(lambda x: x.nlargest(10, 'counts')).reset_index(drop=True)

        fig2a = px.sunburst(sun_data, path=['protocol', 'dst_port'], values='counts',
                           title="Protocol and Port Hierarchy — Full Capture",
                           color='protocol',
                           color_discrete_map={'TCP':'blue', 'UDP':'red', 'ICMP':'orange', 'OTHER':'grey'},
                           template="plotly_dark")
        fig2a.write_html(os.path.join(output_dir, 'charts/protocol_ports/protocol_sunburst.html'))
        fig2a.show()
    except Exception as e: print(f"Error Chart 2A: {e}")

    # --- CHART 2B: TOP 20 TARGETED PORTS ---
    try:
        top_ports = df.groupby(['dst_port', 'protocol']).size().reset_index(name='count')
        top_ports = top_ports.sort_values('count', ascending=False).head(20)

        # Add Service Name Labels
        service_map = {30120: 'FiveM', 53: 'DNS', 80: 'HTTP', 443: 'HTTPS', 22: 'SSH', 19132: 'Minecraft'}
        top_ports['label'] = top_ports['dst_port'].apply(lambda x: f"{x} ({service_map.get(x, 'Unknown')})")

        # Define colors (Gold for 30120)
        colors = []
        for _, row in top_ports.iterrows():
            if row['dst_port'] == 30120: colors.append('gold')
            elif row['protocol'] == 'UDP': colors.append('red')
            elif row['protocol'] == 'TCP': colors.append('blue')
            else: colors.append('orange')

        fig2b = px.bar(top_ports, x='count', y='label', orientation='h', text='count',
                       title="Top 20 Targeted Ports", template="plotly_dark",
                       color_discrete_sequence=[colors], color='label')
        fig2b.update_layout(showlegend=False)
        fig2b.write_html(os.path.join(output_dir, 'charts/protocol_ports/top_ports_bar.html'))
        fig2b.show()
    except Exception as e: print(f"Error Chart 2B: {e}")

    # --- CHART 2C: SOURCE PORT VS DEST PORT SCATTER ---
    try:
        sample_df = df.sample(n=min(50000, len(df)))
        fig2c = px.scatter(sample_df, x='src_port', y='dst_port', color='protocol',
                          size='packet_length', hover_data=['src_ip', 'dst_ip'],
                          title="Source Port vs Destination Port Scatter (Random Sample)",
                          template="plotly_dark", opacity=0.4)
        fig2c.write_html(os.path.join(output_dir, 'charts/protocol_ports/port_scatter.html'))
        fig2c.show()
    except Exception as e: print(f"Error Chart 2C: {e}")

    # --- CHART 2D: PROTOCOL BAR RACE (ANIMATED) ---
    try:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['minute'] = df['timestamp'].dt.strftime('%H:%M')
        race_data = df.groupby(['minute', 'protocol']).size().reset_index(name='count')

        fig2d = px.bar(race_data, x="count", y="protocol", color="protocol",
                       animation_frame="minute", orientation='h', range_x=[0, race_data['count'].max()*1.1],
                       title="Protocol Distribution Race — Minute by Minute",
                       template="plotly_dark",
                       color_discrete_map={'TCP':'blue', 'UDP':'orange', 'ICMP':'red', 'OTHER':'grey'})
        fig2d.write_html(os.path.join(output_dir, 'charts/protocol_ports/protocol_race.html'))
        fig2d.show()
    except Exception as e: print(f"Error Chart 2D: {e}")

run_module_2(PARSED_CSV_PATH, BASE_OUTPUT_DIR)

Output hidden; open in https://colab.research.google.com to view.

## Visual Module 3: Packet Behavior (TTL, Flags, Payload)
This module dives into packet headers and payloads to identify attack tools and spoofing.
- **TTL Analysis**: Helps determine if the traffic is likely spoofed by checking for inconsistent or unusual Time-To-Live values.
- **TCP Flag Radar**: Quickly identifies the type of TCP flood (e.g., a massive SYN spike indicating a SYN flood).
- **Payload Violin**: Visualizes the distribution of packet sizes to identify amplification or standard flooding patterns.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os

def run_module_3(csv_path, output_dir):
    df = pd.read_csv(csv_path)

    # --- CHART 3A: TTL DISTRIBUTION ---
    try:
        ttl_counts = df['ttl'].value_counts().reset_index(name='count')
        ttl_counts.columns = ['ttl', 'count']

        fig3a = px.bar(ttl_counts, x='ttl', y='count', title='TTL Value Distribution — Spoofing Detection',
                       template='plotly_dark', color='ttl', color_continuous_scale='RdYlGn_r')
        fig3a.add_vrect(x0=63, x1=65, fillcolor='green', opacity=0.2, annotation_text='Linux/Mac')
        fig3a.add_vrect(x0=127, x1=128, fillcolor='blue', opacity=0.2, annotation_text='Windows')
        fig3a.write_html(os.path.join(output_dir, 'charts/packet_behavior/ttl_histogram.html'))
        fig3a.show()
    except Exception as e: print(f'Error Chart 3A: {e}')

    # --- CHART 3B: TCP FLAG RADAR ---
    try:
        tcp_df = df[df['protocol'] == 'TCP']
        if not tcp_df.empty:
            flags_series = tcp_df['tcp_flags'].dropna().str.split('|').explode()
            flags_counts = flags_series.value_counts().reset_index()
            flags_counts.columns = ['flag', 'count']

            fig3b = go.Figure(data=go.Scatterpolar(r=flags_counts['count'], theta=flags_counts['flag'], fill='toself'))
            fig3b.update_layout(polar=dict(radialaxis=dict(visible=True)), title='TCP Flag Radar Chart', template='plotly_dark')
            fig3b.write_html(os.path.join(output_dir, 'charts/packet_behavior/tcp_flag_radar.html'))
            fig3b.show()
    except Exception as e: print(f'Error Chart 3B: {e}')

    # --- CHART 3C: PAYLOAD VIOLIN (COMPRESSED) ---
    try:
        # Sampling to 50k points and using CDN to fix GitHub upload issues
        sample_size = min(50000, len(df))
        df_sampled = df.sample(n=sample_size, random_state=42)

        fig3c = px.violin(df_sampled, y='payload_size', x='protocol', box=True, points=None,
                         title=f'Payload Size Distribution (Sampled {sample_size:,} pkts)', template='plotly_dark')
        fig3c.add_hline(y=512, line_dash='dash', annotation_text='DNS Amp Threshold')
        fig3c.add_hline(y=1400, line_dash='dash', annotation_text='MTU Bound')

        fig3c.update_layout(hovermode=False)

        output_path = os.path.join(output_dir, 'charts/packet_behavior/payload_violin.html')
        # include_plotlyjs='cdn' reduces file size by ~3MB
        fig3c.write_html(output_path, include_plotlyjs='cdn')
        print(f'Compressed chart saved to: {output_path}')
        fig3c.show()
    except Exception as e: print(f'Error Chart 3C: {e}')

run_module_3(PARSED_CSV_PATH, BASE_OUTPUT_DIR)

Compressed chart saved to: /content/drive/MyDrive/DDoS_Visual_Analysis/charts/packet_behavior/payload_violin.html


## Visual Module 4: Attacker IP Origins (Geo & Map)
This module maps the physical location of the attackers.
1. **Enrichment**: We use MaxMind GeoLite2 and IPWhois to find the Country, City, and ISP of the top 500 attacking IPs.
2. **Heatmap**: Visualizes the global concentration of the botnet.
3. **Marker Map**: Provides a detailed look at individual high-volume attackers, including their ISP information.

In [ ]:
import pandas as pd
import geoip2.database
from ipwhois import IPWhois
import folium
from folium.plugins import HeatMap, Fullscreen
import plotly.express as px
import os
from tqdm import tqdm

def run_module_4(csv_path, geoip_db, output_dir):
    df = pd.read_csv(csv_path)
    top_ips = df['src_ip'].value_counts().head(500).reset_index()
    top_ips.columns = ['ip', 'packet_count']

    geo_results = []
    print("Enriching top 500 IPs with Geo and ASN data...")

    try:
        reader = geoip2.database.Reader(geoip_db)
    except Exception as e:
        print(f"GeoIP Database error: {e}. Please ensure the path is correct.")
        return pd.DataFrame() # Return an empty DataFrame on error

    for _, row in tqdm(top_ips.iterrows(), total=len(top_ips)):
        ip = row['ip']
        data = {'ip': ip, 'packet_count': row['packet_count'], 'lat': None, 'lon': None, 'country': 'Unknown', 'city': 'Unknown', 'isp': 'Unknown'}

        try:
            response = reader.city(ip)
            data['lat'] = response.location.latitude
            data['lon'] = response.location.longitude
            data['country'] = response.country.name
            data['city'] = response.city.name
        except:
            # If geo-lookup fails, lat/lon/country/city remain None/Unknown
            pass

        # IPWhois lookup for ISP (ASN)
        try:
            obj = IPWhois(ip)
            results = obj.lookup_rdap(depth=1)
            if results and 'asn_description' in results:
                data['isp'] = results['asn_description']
        except:
            pass # Keep 'Unknown' if lookup fails

        geo_results.append(data)

    reader.close()
    enriched_df = pd.DataFrame(geo_results)

    # Convert lat/lon to numeric, forcing errors to NaN for robust dropna
    enriched_df['lat'] = pd.to_numeric(enriched_df['lat'], errors='coerce')
    enriched_df['lon'] = pd.to_numeric(enriched_df['lon'], errors='coerce')

    enriched_df.to_csv(os.path.join(output_dir, 'data/ip_geo_enriched.csv'), index=False)

    # --- CHART 4A: GLOBAL HEATMAP ---
    try:
        # Filter out entries where latitude or longitude is NaN
        heatmap_filtered_df = enriched_df.dropna(subset=['lat', 'lon'])

        if not heatmap_filtered_df.empty:
            m1 = folium.Map(location=[20, 0], zoom_start=2, tiles='CartoDB dark_matter')
            heat_data = [[row['lat'], row['lon'], row['packet_count']] for _, row in heatmap_filtered_df.iterrows()]
            HeatMap(heat_data).add_to(m1)
            Fullscreen().add_to(m1)
            m1.save(os.path.join(output_dir, 'charts/geo_map/attack_heatmap_world.html'))
        else:
            print("Skipping Heatmap: No valid geo-location data after filtering.")
    except Exception as e: print(f"Error Chart 4A: {e}")

    # --- CHART 4B: ISP TREEMAP ---
    try:
        # Filter out entries where country or isp is None or 'Unknown'
        treemap_filtered_df = enriched_df[
            enriched_df['country'].notna() & (enriched_df['country'] != 'Unknown') &
            enriched_df['isp'].notna() & (enriched_df['isp'] != 'Unknown')
        ]

        if not treemap_filtered_df.empty:
            fig4b = px.treemap(treemap_filtered_df, path=['country', 'isp'], values='packet_count',
                              title="Attack Source ISP / ASN Treemap", template="plotly_dark")
            fig4b.write_html(os.path.join(output_dir, 'charts/geo_map/isp_treemap.html'))
            fig4b.show()
        else:
            print("Skipping Treemap: No valid country/ISP data after filtering.")
    except Exception as e: print(f"Error Chart 4B: {e}")

    return enriched_df


### Retrying Visual Module 4 with newly downloaded database...

In [ ]:
enriched_df = run_module_4(PARSED_CSV_PATH, GEOIP_DB_PATH, BASE_OUTPUT_DIR)

Enriching top 500 IPs with Geo and ASN data...


100%|██████████| 162/162 [02:07<00:00,  1.27it/s]


In [ ]:
print('Displaying the first 5 rows of enriched_df:')
display(enriched_df.head())

Displaying the first 5 rows of enriched_df:


NameError: name 'enriched_df' is not defined

In [ ]:
print('\nDisplaying info about enriched_df:')
display(enriched_df.info())


Displaying info about enriched_df:


NameError: name 'enriched_df' is not defined

## Final Forensic Summary
This section aggregates the key metrics discovered across all visual modules.

In [ ]:
import pandas as pd

def generate_summary(parsed_csv, enriched_csv):
    df = pd.read_csv(parsed_csv)
    geo_df = pd.read_csv(enriched_csv)

    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['second'] = df['timestamp'].dt.floor('s')
    timeline = df.groupby('second').size()

    peak_pps = timeline.max()
    total_packets = len(df)
    top_proto = df['protocol'].value_counts().idxmax()
    top_port = df['dst_port'].value_counts().idxmax()

    # Extended summary information
    total_unique_ips = df['src_ip'].nunique()
    top_attack_ips_df = df['src_ip'].value_counts().head(5).reset_index()
    top_attack_ips_df.columns = ['ip', 'count']
    top_ips_list = top_attack_ips_df.to_dict('records')

    # Geo Stats - finding the country and ISP with the most packet counts
    top_country_summary = geo_df['country'].value_counts().head(1)
    top_country = top_country_summary.index[0] if not top_country_summary.empty else 'Unknown'
    top_country_count = int(top_country_summary.values[0]) if not top_country_summary.empty else 0

    top_isp_summary = geo_df['isp'].value_counts().head(1)
    top_isp = top_isp_summary.index[0] if not top_isp_summary.empty else 'Unknown'
    top_isp_count = int(top_isp_summary.values[0]) if not top_isp_summary.empty else 0

    summary_results = {
        'total_packets': f"{total_packets:,}",
        'peak_pps': f"{peak_pps:,} PPS",
        'primary_attack_protocol': top_proto,
        'primary_attack_port': top_port,
        'total_unique_ips': f"{total_unique_ips:,}",
        'top_attacking_ips': top_ips_list,
        'top_attacker_origin_country': f"{top_country} ({top_country_count:,} packets)",
        'top_attacker_origin_isp': f"{top_isp} ({top_isp_count:,} packets)"
    }

    print("--- DDOS FORENSIC REPORT SUMMARY ---")
    for key, value in summary_results.items():
        print(f"{key.replace('_', ' ').title()}: {value}")
    print(f"\nAll interactive HTML reports are saved in: {BASE_OUTPUT_DIR}charts/")

    return summary_results

summary_info = generate_summary(PARSED_CSV_PATH, ENRICHED_CSV_PATH)

--- DDOS FORENSIC REPORT SUMMARY ---
Total Packets: 18,301,452
Peak Pps: 152,376 PPS
Primary Attack Protocol: UDP
Primary Attack Port: 1222
Total Unique Ips: 162
Top Attacking Ips: [{'ip': '35.200.213.57', 'count': 7822113}, {'ip': '34.100.137.247', 'count': 5236039}, {'ip': '65.20.74.174', 'count': 2300915}, {'ip': '34.126.187.132', 'count': 1110872}, {'ip': '45.116.228.118', 'count': 387879}]
Top Attacker Origin Country: United States (49 packets)
Top Attacker Origin Isp: Unknown (14 packets)

All interactive HTML reports are saved in: /content/drive/MyDrive/DDoS_Visual_Analysis/charts/


## Visual Dashboard: Consolidated Forensic Report
This cell generates a single `index.html` dashboard to organize all findings in one place.

In [ ]:
import os

def create_enhanced_dashboard(output_dir, summary_data):
    dashboard_path = os.path.join(output_dir, 'index.html')

    # Format top attacking IPs for display
    top_ips_formatted = ''
    for ip_data in summary_data['top_attacking_ips']:
        top_ips_formatted += f"<li><span class='accent'>{ip_data['ip']}</span> ({ip_data['count']:,} packets)</li>"

    html_content = f"""
    <!DOCTYPE html>
    <html lang='en'>
    <head>
        <meta charset='UTF-8'>
        <meta name='viewport' content='width=device-width, initial-scale=1.0'>
        <title>DDoS Visual Forensic Analysis Report</title>
        <link href='https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600;700&display=swap' rel='stylesheet'>
        <style>
            body {{ font-family: 'Inter', sans-serif; background-color: #0f172a; color: #f8fafc; margin: 0; line-height: 1.6; display: flex; min-height: 100vh; }}
            .sidebar {{ position: fixed; left: 0; top: 0; width: 250px; height: 100vh; background: #1e293b; padding: 20px; border-right: 1px solid #334155; box-shadow: 2px 0 5px rgba(0,0,0,0.2); flex-shrink: 0; overflow-y: auto; }}
            .main {{ margin-left: 270px; padding: 40px; flex-grow: 1; max-width: calc(100% - 270px); box-sizing: border-box; }}
            .card {{ background: #1e293b; padding: 25px; border-radius: 12px; margin-bottom: 30px; border: 1px solid #334155; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1), 0 2px 4px -1px rgba(0,0,0,0.06); }}
            .accent {{ color: #38bdf8; font-weight: 600; }}
            .warning {{ border-left: 5px solid #ef4444; }}
            .info {{ border-left: 5px solid #3b82f6; }}
            .success {{ border-left: 5px solid #10b981; }}
            h1 {{ font-size: 2.8rem; color: #e2e8f0; margin-bottom: 10px; font-weight: 700; }}
            h2 {{ font-size: 1.8rem; color: #38bdf8; border-bottom: 1px solid #334155; padding-bottom: 10px; margin-top: 40px; margin-bottom: 25px; font-weight: 600; }}
            h3 {{ color: #f8fafc; font-size: 1.2rem; margin-top: 0; margin-bottom: 15px; font-weight: 600; }}
            h4 {{ color: #94a3b8; font-size: 0.9rem; text-transform: uppercase; letter-spacing: 0.05em; margin-bottom: 10px; font-weight: 600; }}
            iframe {{ width: 100%; height: 550px; border: none; border-radius: 8px; background: #ffffff; margin-top: 15px; box-shadow: inset 0 2px 4px rgba(0,0,0,0.06); }}
            .talking-points {{ background: #0f172a; padding: 20px; border-radius: 8px; margin-top: 25px; border: 1px dashed #475569; }}
            ul {{ list-style-type: disc; padding-left: 25px; margin-top: 10px; }}
            ol {{ list-style-type: decimal; padding-left: 25px; margin-top: 10px; }}
            li {{ margin-bottom: 8px; color: #cbd5e1; }}
            a {{ color: #38bdf8; text-decoration: none; }}
            a:hover {{ text-decoration: underline; }}
            .footer {{ margin-top: 60px; padding-top: 20px; border-top: 1px solid #334155; color: #64748b; font-size: 0.85rem; text-align: center; }}
            .summary-item {{ margin-bottom: 10px; }}
        </style>
    </head>
    <body>
        <div class='sidebar'>
            <h3><span class='accent'>DDoS</span> Analysis</h3>
            <p><a href='#summary'>1. Executive Summary</a></p>
            <p><a href='#key-findings'>2. Key Findings</a></p>
            <p><a href='#methodology'>3. Analysis Methodology</a></p>
            <p><a href='#timeline'>4. Traffic Timeline</a></p>
            <p><a href='#protocols'>5. Protocols & Ports</a></p>
            <p><a href='#geo-location'>6. Geo-Location & Origins</a></p>
            <p><a href='#packet-behavior'>7. Packet Behavior</a></p>
        </div>

        <div class='main'>
            <h1>DDoS Visual Forensic Analysis Report</h1>
            <p class='accent'>Target: FiveM Gameserver | Incident Date: August 17, 2022</p>

            <section id='summary' class='card warning'>
                <h2>1. Executive Summary</h2>
                <p>A sophisticated and high-intensity Distributed Denial of Service (DDoS) attack was observed targeting a FiveM game server. This report provides a detailed forensic breakdown of the incident, identifying key attack vectors, origins, and patterns.</p>
                <p class='summary-item'><strong>Total Packets Analyzed:</strong> <span class='accent'>{summary_data['total_packets']}</span></p>
                <p class='summary-item'><strong>Peak Traffic Intensity:</strong> <span class='accent'>{summary_data['peak_pps']}</span></p>
                <p class='summary-item'><strong>Primary Attack Vector:</strong> <span class='accent'>{summary_data['primary_attack_protocol']}</span> flood targeting Destination Port <span class='accent'>{summary_data['primary_attack_port']}</span>.</p>
                <p class='summary-item'><strong>Top Attacker Origin (Country):</strong> <span class='accent'>{summary_data['top_attacker_origin_country']}</span></p>
                <p class='summary-item'><strong>Top Attacker Origin (ISP):</strong> <span class='accent'>{summary_data['top_attacker_origin_isp']}</span></p>
            </section>

            <section id='key-findings' class='card info'>
                <h2>2. Key Findings</h2>
                <ul>
                    <li>A total of <span class='accent'>{summary_data['total_packets']}</span> packets were captured and analyzed over the incident period.</li>
                    <li>The attack reached a peak intensity of <span class='accent'>{summary_data['peak_pps']}</span>.</li>
                    <li>The primary attack method was identified as a <span class='accent'>{summary_data['primary_attack_protocol']}</span> flood, specifically targeting port <span class='accent'>{summary_data['primary_attack_port']}</span>, which is consistent with common FiveM server attack vectors.</li>
                    <li><span class='accent'>{summary_data['total_unique_ips']}</span> unique source IP addresses were involved in the attack, suggesting a distributed botnet.</li>
                    <li>The top 5 attacking IP addresses by packet count are:
                        <ul>
                            {top_ips_formatted}
                        </ul>
                    </li>
                    <li>Geographic analysis points to <span class='accent'>{summary_data['top_attacker_origin_country']}</span> as the most significant origin country for the attack traffic, often routed through ISPs like <span class='accent'>{summary_data['top_attacker_origin_isp']}</span>.</li>
                    <li>Packet behavior analysis indicates widespread IP spoofing, a common tactic to obscure attacker identity.</li>
                </ul>
            </section>

            <section id='methodology' class='card success'>
                <h2>3. Analysis Methodology</h2>
                <p>Our forensic investigation employed a multi-stage data engineering and visualization pipeline:</p>
                <ol>
                    <li><strong>Packet Extraction:</strong> Raw binary PCAP data was stream-parsed using the <code>dpkt</code> library, extracting critical header information including IP addresses, TCP/UDP flags, and Time-To-Live (TTL) values.</li>
                    <li><strong>Data Normalization:</strong> Timestamps were converted to a uniform format, and traffic metrics such as Packets Per Second (PPS) and Megabits Per Second (Mbps) were computed for temporal analysis.</li>
                    <li><strong>Geo-Enrichment:</strong> Unique source IP addresses were cross-referenced against the <code>MaxMind GeoLite2</code> database for geographical mapping (Country, City, Latitude, Longitude) and `IPWhois` for Autonomous System Number (ASN)/ISP identification.</li>
                    <li><strong>Visual Synthesis:</strong> Interactive visualizations were generated using <code>Plotly</code> and <code>Folium</code> to identify attack patterns, traffic anomalies, and geographical distribution of attack sources.</li>
                </ol>
            </section>

            <section id='timeline'>
                <h2>4. Traffic Timeline & Attack Pulse</h2>
                <p>This section illustrates the temporal characteristics of the attack, highlighting its start, peak intensity, and duration.</p>
                <iframe src='charts/timeline/attack_timeline.html' title='Attack Traffic Timeline'></iframe>
                <div class='talking-points'>
                    <h4>Analysis Highlights:</h4>
                    <ul>
                        <li>Observe the sudden and significant increase in PPS and Mbps, indicating the precise commencement of the DDoS event.</li>
                        <li>The comparison of PPS and Mbps reveals the volumetric nature of the attack, showcasing both packet frequency and bandwidth consumption.</li>
                        <li>Protocol stacking over time (e.g., UDP dominance) confirms the primary attack vector throughout the incident.</li>
                    </ul>
                </div>
                <iframe src='charts/timeline/protocol_stacked_area.html' title='Protocol Breakdown Over Time'></iframe>
                <div class='talking-points'>
                    <h4>Analysis Highlights:</h4>
                    <ul>
                        <li>The stacked area chart clearly shows the evolution of attack protocols.</li>
                        <li>Dominance of <span class='accent'>{summary_data['primary_attack_protocol']}</span> confirms it as the primary weapon used in this attack.</li>
                    </ul>
                </div>
                <iframe src='charts/timeline/attack_heatmap.html' title='Attack Wave Heatmap'></iframe>
                <div class='talking-points'>
                    <h4>Analysis Highlights:</h4>
                    <ul>
                        <li>The heatmap visualizes packet distribution across seconds and minutes, revealing if the attack was continuous or bursty.</li>
                        <li>Dense red areas indicate high packet rates, pinpointing intense attack periods.</li>
                    </ul>
                </div>
            </section>

            <section id='protocols'>
                <h2>5. Protocol & Port Target Analysis</h2>
                <p>Understanding which protocols and ports were targeted is crucial for identifying the nature of the attack and vulnerable services.</p>
                <iframe src='charts/protocol_ports/protocol_sunburst.html' title='Protocol and Port Hierarchy'></iframe>
                <div class='talking-points'>
                    <h4>Analysis Highlights:</h4>
                    <ul>
                        <li>The sunburst chart provides a hierarchical view of protocol and destination port distribution.</li>
                        <li>It distinctly shows the heavy concentration of traffic on port <span class='accent'>{summary_data['primary_attack_port']}</span> under the <span class='accent'>{summary_data['primary_attack_protocol']}</span> protocol.</li>
                    </ul>
                </div>
                <iframe src='charts/protocol_ports/top_ports_bar.html' title='Top 20 Targeted Ports'></iframe>
                <div class='talking-points'>
                    <h4>Analysis Highlights:</h4>
                    <ul>
                        <li>The bar chart confirms port <span class='accent'>{summary_data['primary_attack_port']}</span> as the most targeted, indicating a direct assault on the FiveM server service.</li>
                        <li>Presence of other ports could suggest secondary reconnaissance or collateral damage.</li>
                    </ul>
                </div>
                <iframe src='charts/protocol_ports/port_scatter.html' title='Source Port vs Destination Port Scatter'></iframe>
                <div class='talking-points'>
                    <h4>Analysis Highlights:</h4>
                    <ul>
                        <li>A scattered distribution of source ports with a consistent destination port indicates IP spoofing.</li>
                        <li>A tight cluster of both source and destination ports might suggest legitimate, but overwhelmed, connections.</li>
                    </ul>
                </div>
                <iframe src='charts/protocol_ports/protocol_race.html' title='Protocol Distribution Race'></iframe>
                <div class='talking-points'>
                    <h4>Analysis Highlights:</h4>
                    <ul>
                        <li>The animated bar race illustrates the minute-by-minute shift in dominant protocols, revealing potential attacker strategy changes.</li>
                    </ul>
                </div>
            </section>

            <section id='geo-location'>
                <h2>6. Global Botnet Distribution (Geo-Location & Origins)</h2>
                <p>Mapping the physical origin of the attack traffic helps in understanding the scope and source of the botnet infrastructure.</p>
                <iframe src='charts/geo_map/attack_heatmap_world.html' title='Global Attack Heatmap'></iframe>
                <div class='talking-points'>
                    <h4>Analysis Highlights:</h4>
                    <ul>
                        <li>The global heatmap visually identifies geographical clusters of attacking IP addresses.</li>
                        <li>Dense areas indicate significant contributions from specific regions, with <span class='accent'>{summary_data['top_attacker_origin_country']}</span> being a prominent origin.</li>
                    </ul>
                </div>
                <iframe src='charts/geo_map/isp_treemap.html' title='Attack Source ISP / ASN Treemap'></iframe>
                <div class='talking-points'>
                    <h4>Analysis Highlights:</h4>
                    <ul>
                        <li>The treemap details the distribution of attack traffic by country and Internet Service Provider (ISP) / Autonomous System Number (ASN).</li>
                        <li>Large blocks represent major contributing ISPs, such as <span class='accent'>{summary_data['top_attacker_origin_isp']}</span>, highlighting common hosting or proxy services used by attackers.</li>
                    </ul>
                </div>
            </section>

            <section id='packet-behavior'>
                <h2>7. Packet Behavior (TTL, Flags, Payload)</h2>
                <p>Examining individual packet characteristics can reveal further insights into the attack methods, including spoofing and tool identification.</p>
                <iframe src='charts/packet_behavior/ttl_histogram.html' title='TTL Value Distribution'></iframe>
                <div class='talking-points'>
                    <h4>Analysis Highlights:</h4>
                    <ul>
                        <li>The Time-To-Live (TTL) distribution is critical for detecting IP spoofing. Inconsistent or unusual TTL values, deviating from standard OS defaults (e.g., 64, 128), strongly suggest spoofed source IPs.</li>
                    </ul>
                </div>
                <iframe src='charts/packet_behavior/tcp_flag_radar.html' title='TCP Flag Radar Chart'></iframe>
                <div class='talking-points'>
                    <h4>Analysis Highlights:</h4>
                    <ul>
                        <li>For TCP-based attacks, the radar chart visualizes the frequency of different TCP flags (SYN, ACK, RST, FIN).</li>
                        <li>A disproportionate number of SYN flags, for instance, is a hallmark of a SYN flood attack.</li>
                    </ul>
                </div>
                <iframe src='charts/packet_behavior/payload_violin.html' title='Payload Size Distribution by Protocol'></iframe>
                <div class='talking-points'>
                    <h4>Analysis Highlights:</h4>
                    <ul>
                        <li>The violin plot shows the distribution of packet payload sizes across different protocols.</li>
                        <li>Anomalously large UDP payloads, especially exceeding typical DNS response sizes (e.g., 512 bytes), can indicate amplification attacks.</li>
                    </ul>
                </div>
            </section>

            <footer class='footer'>
                Generated by DDoS Visual Analysis Module | Forensic Lead Analysis Team
            </footer>
        </div>
    </body>
    </html>
    """

    with open(dashboard_path, 'w') as f:
        f.write(html_content)
    print(f"Professional Forensic Presentation created at: {dashboard_path}")

create_enhanced_dashboard(BASE_OUTPUT_DIR, summary_info)

Professional Forensic Presentation created at: /content/drive/MyDrive/DDoS_Visual_Analysis/index.html


## Documentation: Repository README.md
This cell generates a `README.md` file for your GitHub repository to explain the project to others.

In [ ]:
import os

readme_content = """# DDoS Visual Forensic Analysis: FiveM Server Attack

## Project Overview
This repository contains a comprehensive forensic analysis pipeline for investigating Distributed Denial of Service (DDoS) attacks using raw packet captures (`.pcapng`). Using a real-world dataset from a FiveM game server attack (August 2022), this project transforms millions of raw packets into an interactive, presentation-ready forensic dashboard.

### Key Features
- **High-Performance Parsing**: Stream-parsing of large PCAP files using `dpkt`.
- **Interactive Visualizations**: Dynamic charts for traffic pulses, protocol distribution, and port targeting using `Plotly`.
- **Geographical Intelligence**: Mapping attack origins using `MaxMind GeoLite2` and `Folium` heatmaps.
- **Automated Dashboard**: Generates a consolidated `index.html` report with presentation talking points for faculty reviews.

## Analysis Methodology
We follow a 4-stage forensic data engineering process:
1. **Extraction**: Decoding binary packet data to extract IP headers, TCP/UDP flags, and TTL values.
2. **Normalization**: Scaling raw timestamps into PPS (Packets Per Second) and Mbps (Megabits Per Second) metrics.
3. **Enrichment**: Augmenting source IP addresses with Geo-location (Country, City) and ASN (ISP) data.
4. **Synthesis**: Correlating packet behavior (like TTL variance) with geographical clusters to identify botnet footprints.

## Forensic Modules
- **Module 1: Attack Pulse**: Visualizes the start, peak, and duration of the attack.
- **Module 2: Protocol Sunburst**: Identifies the primary attack vector (e.g., UDP Flood on Port 1222).
- **Module 3: Global Heatmap**: Shows the physical distribution of the attacking botnet.
- **Module 4: TTL Fingerprinting**: Uses Time-to-Live values to detect IP spoofing and distance of attackers.

## Getting Started
1. **Data**: Place your `.pcapng` file in the designated path.
2. **Database**: Download the `GeoLite2-City.mmdb` from MaxMind.
3. **Run**: Execute the Jupyter/Colab notebook to generate the `charts/` and the final `index.html`.

## Visual Report Preview
The final output is an interactive dashboard that organizes all charts with built-in "Faculty Presentation Points" to assist in technical explanations.

---
*Note: This project is intended for educational and forensic research purposes.*
"""

readme_path = os.path.join(BASE_OUTPUT_DIR, 'README.md')
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(readme_content)

print(f"README.md created successfully at: {readme_path}")

README.md created successfully at: /content/drive/MyDrive/DDoS_Visual_Analysis/README.md


## Deployment: Push to GitHub
Run the following cell to initialize your repository and prepare the files for upload. **Note:** You will need to replace `YOUR_GITHUB_REPO_URL` with your actual repository link.

In [ ]:
import os

# --- SETTINGS ---
GITHUB_REPO_URL = 'https://github.com/ERROR-SIDDH/DDoS_Visual_Analysis.git' # <--- UPDATE THIS
USER_EMAIL = 'sidharthmannikoth@yahoo.com'
USER_NAME = 'Sidharth-M'

def push_to_github():
    # Move to the project directory
    os.chdir(BASE_OUTPUT_DIR)

    # Initialize and configure git
    !git init
    !git config --global user.email "{USER_EMAIL}"
    !git config --global user.name "{USER_NAME}"

    # Add files and commit
    !git add .
    !git commit -m "Initial commit: Professional DDoS Forensic Analysis Dashboard"

    # Push (Note: This may require a Personal Access Token if not already authenticated)
    if 'YOUR_GITHUB_REPO_URL' not in GITHUB_REPO_URL:
        !git remote add origin {GITHUB_REPO_URL}
        !git branch -M main
        print("\nReady to push. Run: !git push -u origin main")
    else:
        print("Please update the GITHUB_REPO_URL variable with your repository link.")

push_to_github()

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/drive/MyDrive/DDoS_Visual_Analysis/.git/
[master (root-commit) c5e8b8f] Initial commit: Professional DDoS Forensic Analysis Dashboard
 16 files changed, 684 insertions(+)
 create mode 100644 README.md
 create mode 100644 charts/geo_map/attack_heatmap_world.html
 create mode 100644 charts/geo_map/isp_treemap.html
 create mode 100644 charts/packet_behavior/payload_violin.html
 create mode 100644 charts/packet_behavior/tcp_flag_radar.html
 create mode 100644 charts/pac

In [ ]:
import getpass
import subprocess
import os

os.chdir(BASE_OUTPUT_DIR)

# 1. Forcefully clear working directory (stash untracked files)
!git add .
!git commit -m "Final pre-purge checkpoint"
!git stash -u

# 2. Purge the 'data/' folder from Git history
print("Purging 'data/' folder from repository history... This may take a moment.")
!git filter-branch --force --index-filter 'git rm -rf --cached --ignore-unmatch data/' --prune-empty --tag-name-filter cat -- --all

# 3. Cleanup local repo
!rm -rf .git/refs/original/
!git reflog expire --expire=now --all
!git gc --prune=now --aggressive

# 4. Detect branch
try:
    current_branch = subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']).decode('utf-8').strip()
except:
    current_branch = 'main'

# 5. Auth and Push
github_user = input('Enter your GitHub username: ')
github_token = getpass.getpass('Enter your GitHub Personal Access Token: ')
repo_url = 'github.com/ERROR-SIDDH/DDoS_Visual_Analysis.git'

authenticated_url = f'https://{github_user}:{github_token}@{repo_url}'
!git remote set-url origin {authenticated_url}

print(f"Pushing clean project to origin/{current_branch}...")
!git push -f -u origin {current_branch}

On branch main
nothing to commit, working tree clean
No local changes to save
Purging 'data/' folder from repository history... This may take a moment.
	 rewrites.  Hit Ctrl-C before proceeding to abort, then use an
	 alternative filtering tool such as 'git filter-repo'
	 (https://github.com/newren/git-filter-repo/) instead.  See the
	 filter-branch manual page for more details; to squelch this warning,
	 set FILTER_BRANCH_SQUELCH_WARNING=1.
Proceeding with filter-branch...

Rewrite c5e8b8fe3445d4c825dfbfd986975617c54bbc09 (1/5) (0 seconds passed, remaining 0 predicted)    rm 'data/ip_geo_enriched.csv'
rm 'data/parsed_flows.csv'
Rewrite 5b2f389df7ec69cdf81412d493321b952bea71e8 (2/5) (0 seconds passed, remaining 0 predicted)    rm 'data/ip_geo_enriched.csv'
Rewrite 6a63bc270c7f60b7c285f7bb2a6c3a88020a0e6c (3/5) (0 seconds passed, remaining 0 predicted)    rm 'data/ip_geo_enriched.csv'
Rewrite 95edb214b73134b3e4ed28177e1be75c7f0694c8 (4/5) (1 seconds passed, remaining 0 predicted)    
Re